# Ejercicio 7 — Bloqueo y confusión en un diseño $3^2$ (aritmética modular)

**Objetivo.** Construir un diseño $3^2$ ejecutado en 3 bloques (lotes de materia prima)
usando la relación de confusión modular $L = x_1 + 2x_2 \pmod 3$ (teoría §7.1), medir la
ganancia de precisión al remover del error la variabilidad entre lotes, y entender qué se
sacrifica (la interacción) al elegir este esquema de confusión.

**Factores:**
- $A$ = Tiempo de curado: 10 (−1), 15 (0), 20 min (+1)
- $B$ = Temperatura de curado: 80 (−1), 100 (0), 120 °C (+1)
- **Bloque:** lote de adhesivo (factor de perturbación, 3 lotes)

**Respuesta:** Resistencia al corte por solape (MPa)

**Dataset:** `../../datos/curado-adhesivo-3k-bloques.csv`

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm

df = pd.read_csv('../../datos/curado-adhesivo-3k-bloques.csv')
print(f'Corridas: {len(df)} (3^2 = 9, repartidas en 3 bloques de 3)')
print(df)

## 1. Verificación del esquema de confusión

El experimento se ejecutó en 3 lotes de adhesivo (bloques). Las 9 combinaciones de
tratamiento se repartieron según la relación de confusión modular de la teoría (§7.1):

$$L = x_1^{\text{Yates}} + 2\,x_2^{\text{Yates}} \pmod 3, \qquad \text{bloque} = L$$

donde $x_i^{\text{Yates}} \in \{0,1,2\}$ es la codificación de Yates ($x_i^{\text{Yates}} = x_i+1$).
Esta regla confunde con los bloques el componente de interacción de mayor orden
(el llamado $AB^2$ en notación de Yates), dejando los efectos principales $A$ y $B$
completamente libres de esa fuente de variación.

In [ ]:
x1_yates = df['x1'] + 1
x2_yates = df['x2'] + 1
L = (x1_yates + 2*x2_yates) % 3

verificacion = pd.DataFrame({'x1': df['x1'], 'x2': df['x2'],
                             'L_calculado': L, 'bloque_dataset': df['bloque']})
print(verificacion)
print(f'\n¿El bloque de cada corrida coincide con L mod 3?  {(L.values == df["bloque"].values).all()}')

## 2. Costo de ignorar el bloque

Si se ignora el lote y se analiza el $3^2$ como si fuera homogéneo, la variabilidad
entre lotes queda atrapada en el término de error, inflándolo.

In [ ]:
modelo_sin_bloque = smf.ols('resistencia ~ x1 + x2 + I(x1**2) + I(x2**2)', data=df).fit()
anova_sin = sm.stats.anova_lm(modelo_sin_bloque, typ=1)
print('ANOVA sin bloque:')
print(anova_sin.round(4))
print(f'\nMSE (sin bloque): {modelo_sin_bloque.mse_resid:.4f}  ({modelo_sin_bloque.df_resid:.0f} gl)')

## 3. Modelo con bloque como factor de bloqueo

Incluimos el bloque como factor categórico. La suma de cuadrados entre bloques se separa
del error, y el error residual debería reducirse notablemente si el lote realmente
introduce una perturbación sistemática.

In [ ]:
modelo_con_bloque = smf.ols('resistencia ~ x1 + x2 + I(x1**2) + I(x2**2) + C(bloque)',
                            data=df).fit()
anova_con = sm.stats.anova_lm(modelo_con_bloque, typ=1)
print('ANOVA con bloque:')
print(anova_con.round(4))
print(f'\nMSE (con bloque): {modelo_con_bloque.mse_resid:.4f}  ({modelo_con_bloque.df_resid:.0f} gl)')

## 4. Comparación de precisión: con y sin bloqueo

In [ ]:
comparacion = pd.DataFrame({
    'Modelo':   ['Sin bloque', 'Con bloque'],
    'MSE error': [modelo_sin_bloque.mse_resid, modelo_con_bloque.mse_resid],
    'gl error':  [modelo_sin_bloque.df_resid, modelo_con_bloque.df_resid],
    'p-valor x1': [anova_sin.loc['x1', 'PR(>F)'], anova_con.loc['x1', 'PR(>F)']],
    'p-valor x2': [anova_sin.loc['x2', 'PR(>F)'], anova_con.loc['x2', 'PR(>F)']],
})
print(comparacion.round(5).to_string(index=False))
print(f"\nEl bloque explica SC={anova_con.loc['C(bloque)','sum_sq']:.3f} "
      f"(p={anova_con.loc['C(bloque)','PR(>F)']:.4f}): el lote sí introduce una "
      f"perturbación sistemática real que vale la pena remover del error.")

## 5. Qué se sacrifica: la interacción confundida con bloques

El esquema de confusión elegido reparte las 9 corridas en 3 bloques usando exactamente
2 grados de libertad tomados del espacio de la interacción $AB$ (4 gl en total). Si
intentamos ajustar la interacción $x_1{:}x_2$ **además** del bloque, el modelo queda casi
saturado (8 parámetros para 9 corridas, solo 1 gl de error): no hay manera fiable de
separar la interacción de la variación entre lotes con este diseño.

In [ ]:
modelo_full = smf.ols('resistencia ~ x1 + x2 + I(x1**2) + I(x2**2) + x1:x2 + C(bloque)',
                      data=df).fit()
print(f'Grados de libertad de error con interacción + bloque: {modelo_full.df_resid:.0f}')
print(modelo_full.summary().tables[1])
print('\nCon un solo grado de libertad de error, el error estándar de x1:x2 es enorme y')
print('su intervalo de confianza no permite concluir nada: la interacción quedó')
print('confundida (parcialmente) con la partición en bloques.')

## 6. Conclusión

- El esquema de confusión $L = x_1 + 2x_2 \pmod 3$ reproduce exactamente la tabla de la
  teoría (§7.1) y reparte las 9 corridas en 3 bloques de 3, dejando los efectos
  principales $A$ y $B$ (lineal y cuadrático) libres de la variación entre lotes.
- Ignorar el bloqueo cuando existe una fuente de variación real (aquí, el lote) infla el
  error: el MSE pasa de $4.04$ a $0.15$ al reconocer el bloque, y los efectos principales
  pasan de ser marginalmente significativos a altamente significativos.
- El precio de esta ganancia es la interacción $AB$: al usarla (parcialmente) para
  definir los bloques, ya no puede estimarse de forma independiente y confiable.
- **Regla práctica.** Bloquea por lote, turno o cualquier factor de perturbación conocido
  siempre que sospeches que introduce variabilidad sistemática; acepta confundir la
  interacción de mayor orden si —como suele ocurrir— es la menos relevante para el
  objetivo del experimento.